# NumCompute quickstart

End-to-end walk through the package: CSV in, preprocessing, statistics, sort/rank, metrics, gradients, pipeline and a benchmark.

In [ ]:
import numpy as np
import sys
sys.path.insert(0, "..")  # Find numcompute sample csv in the parent folder.

from numcompute import io
from numcompute import preprocessing as pp
from numcompute import stats
from numcompute.utils import iter_batches

sample_path = "data/sample.csv"

np.set_printoptions(precision=3, suppress=True)

## 1. Read CSV (`io.py`)

`data/sample.csv` has 3,000 rows: two numeric features (`age`, `income`) and two text columns, the true `city` and a simulated prediction `city_predict` (used in Section 5). Age and income have missing cells written as `""`, `NA` or `NaN`; all of them are in `io.DEFAULT_MISSING`, so `load_csv` treats them as missing by default and numeric columns get NaN (a text column would get `''`). `on_missing='skip'` drops incomplete rows instead, and `iter_csv_chunks` streams the same file in fixed-size chunks.

In [ ]:
X_num, num_names = io.load_csv(sample_path, usecols=['age', 'income'], dtype=float, return_header=True)
city = io.load_csv(sample_path, usecols='city')
X = io.load_csv(sample_path)
print(num_names, X_num.shape, 'missing per column:', np.isnan(X_num).sum(axis=0), '| missing cities:', (city == '').sum())
print(X_num[:5])
print(city[:5].ravel())

print('complete rows only:', io.load_csv(sample_path, usecols=num_names, dtype=float, on_missing='skip').shape)

# same file streamed 500 rows at a time; stacking the chunks gives the same array
chunks = list(io.iter_csv_chunks(sample_path, chunk_size=500, usecols=num_names, dtype=float))
print([c.shape for c in chunks], np.array_equal(np.vstack(chunks), X_num, equal_nan=True))

## 2. Preprocess (`preprocessing.py`)

Impute missing numbers, then scale. Fit on train, apply to new rows with the same statistics.

In [ ]:
imp = pp.SimpleImputer(strategy='median')
X_full = imp.fit_transform(X_num)
print('fill values:', imp.statistics_)

scaler = pp.StandardScaler().fit(X_full)
Z = scaler.transform(X_full)
print('mean after scaling:', Z.mean(axis=0), ' std:', Z.std(axis=0))

print(pp.MinMaxScaler((0, 1)).fit_transform(X_full)[:3])

In [ ]:
enc = pp.OneHotEncoder(handle_unknown='ignore').fit(city)
print('categories:', enc.categories_[0])
print(enc.transform(city[:5]))       # blank city -> all zeros
print(enc.transform([['Darwin']]))   # unseen city, ignored

In [ ]:
# scaler fitted on 2 features rejects a 1-feature matrix with a clear message
try:
    scaler.transform(X_full[:, :1])
except ValueError as e:
    print('ValueError:', e)

## 3. Descriptive statistics (`stats.py`)

`axis=0` summarises each column. `skip_nan=True` ignores NaN; by default NaN propagates like NumPy.

In [ ]:
print('mean   ', stats.mean(X_num, axis=0))
print('mean   ', stats.mean(X_num, axis=0, skip_nan=True), '(skip NaN)')
print('median ', stats.median(X_num, axis=0, skip_nan=True))
print('std    ', stats.std(X_num, axis=0, ddof=1, skip_nan=True))
print('min/max', stats.min(X_num, axis=0, skip_nan=True), stats.max(X_num, axis=0, skip_nan=True))

In [ ]:
print(stats.quantile(X_num, [0.25, 0.5, 0.75], axis=0, skip_nan=True))   # shape (3, n_features)
counts, edges = stats.histogram(X_num[:, 1], bins=4)
print('income histogram:', counts, edges)

In [ ]:
# streaming stats: same numbers as the batch version, O(d) memory
rs = stats.RunningStats(skip_nan=True)
for batch in iter_batches(X_num, 3):
    rs.update(batch)
print('rows seen:', rs.n)
print('running mean:', rs.mean)
print('running std :', rs.std(ddof=1))

## 4. Sort, search, top-k and rank (`sort_search.py`, `rank.py`)

This section uses `X_full`, created from the CSV's `age` and `income` columns after median imputation in section 2. Imputation preserves every CSV row, so the ranks below remain aligned with the original row order. We use income for sorting, selection, searching, and ranking; selection indices are zero-based.

In [ ]:
from numcompute.rank import rank, percentile_rank, percentile

income = X_full[:, 1]
from numcompute.sort_search import (
    stable_sort, lexsort, top_k, quickselect, binary_search,
)
kth = income.size // 2
top_five, top_five_indices = top_k(income, 5, return_indices=True)
income_ranks = rank(income)

print("Rows used from CSV:", income.size)
sorted_income = stable_sort(income)
print("First five sorted incomes:", sorted_income[:5])
print(
    "Stable sort matches NumPy order:",
    np.array_equal(sorted_income, np.sort(income, kind="stable")),
)
print("Five highest incomes:", top_five)
print("Original row indices of highest incomes:", top_five_indices)
print("First rows sorted by age, then income:", lexsort((income, X_full[:, 0]))[:5])
print("Median income by quickselect:", quickselect(income, kth))
insertion_index, found = binary_search(sorted_income, top_five[0])
print("Highest income found at sorted index:", insertion_index, found)
print("Average income ranks for first five CSV rows:", income_ranks[:5])
print(
    "Percentile rank of the highest income:",
    percentile_rank(income, top_five[0]),
)
print("Income quartiles:", percentile(income, [25, 50, 75]))

## 5. Metrics (`metrics.py`)

`city_predict` is a simulated classifier from the data generator: each row gets the city whose age/income profile is closest, so it never sees the true city. The classification metrics compare it with `city`; `mse` scores a simulated income estimate.

### 5.1 Regression: mean squared error

The estimate is the true income plus Gaussian noise (sd $10,000, fixed seed), on the rows with a known income. It is compared with a baseline that always predicts the mean income. RMSE (the square root of `mse`) is in dollars.

In [ ]:
from numcompute import metrics

income = X_num[:, 1][~np.isnan(X_num[:, 1])]   # rows with a known income
rng = np.random.default_rng(0)
income_hat = income + rng.normal(0, 10_000, size=income.size)   # simulated estimate

baseline = np.full_like(income, income.mean())
print('rows with income:', income.size)
print(f"RMSE estimate:      {np.sqrt(metrics.mse(income, income_hat)):>9,.0f}")
print(f"RMSE mean baseline: {np.sqrt(metrics.mse(income, baseline)):>9,.0f}")

### 5.2 Six classes: city vs city_predict

In the confusion matrix, rows are the true city and columns the predicted one. `average=None` returns one score per class in **sorted** label order. `macro` is the plain mean over cities; `weighted` weights each city by its number of rows (500 each here, so it equals `macro`); `micro` pools the counts over cities and equals accuracy.

In [ ]:
city_true = city.ravel()                                          # read in Section 1
city_pred = io.load_csv(sample_path, usecols='city_predict').ravel()
cities = np.unique(city_true)                                     # sorted: the order of average=None

print('accuracy:', round(metrics.accuracy(city_true, city_pred), 3))
print('cities:  ', cities)
print(metrics.confusion_matrix(city_true, city_pred, labels=cities))
print('f1 per city:', metrics.f1(city_true, city_pred, average=None))
for avg in ('macro', 'micro', 'weighted'):
    print(f"f1 {avg:<9}", round(metrics.f1(city_true, city_pred, average=avg), 3))

### 5.3 Binary: Sydney vs the rest

Binary metrics score one positive class: every other city becomes `'other'` and `pos_label='Sydney'` names the positive class. Precision is the share of rows predicted Sydney that really are; recall is the share of true Sydney rows that were found. The F1 matches Sydney's entry in 5.2.

In [ ]:
is_sydney = np.where(city_true == 'Sydney', 'Sydney', 'other')
pred_sydney = np.where(city_pred == 'Sydney', 'Sydney', 'other')

print(metrics.confusion_matrix(is_sydney, pred_sydney, labels=['other', 'Sydney']))
print('precision', round(metrics.precision(is_sydney, pred_sydney, pos_label='Sydney'), 3))
print('recall   ', round(metrics.recall(is_sydney, pred_sydney, pos_label='Sydney'), 3))
print('f1       ', round(metrics.f1(is_sydney, pred_sydney, pos_label='Sydney'), 3))

### 5.4 ROC curve and AUC

ROC uses income as score due to Sydney having the highest income profile: each threshold flags every row with an income at or above it as Sydney. The AUC is the chance that a random Sydney row has a higher income than a random other row (0.5 = guessing, 1.0 = perfect ranking).

In [ ]:
known = ~np.isnan(X_num[:, 1])                    # rows with a known income
fpr, tpr, thresholds = metrics.roc_curve(is_sydney[known], X_num[known, 1], pos_label='Sydney')

print('points on the curve:', thresholds.size)
for i in np.linspace(0, thresholds.size - 1, 6).astype(int):   # a few points, highest threshold first
    print(f"threshold {thresholds[i]:>10,.0f}   fpr {fpr[i]:.2f}   tpr {tpr[i]:.2f}")
print('AUC:', round(metrics.auc(fpr, tpr), 3))

### 5.5 Informative errors

Labels read with `io.load_csv` are text by default. Comparing text labels with integer predictions raises a `TypeError` instead of silently scoring 0.

In [ ]:
city_codes = np.searchsorted(cities, city_pred)   # the same predictions, as integers 0-5
try:
    metrics.accuracy(city_true, city_codes)
except TypeError as err:
    print('TypeError:', err)

## 6. Pipeline (`pipeline.py`)

A Pipeline runs named steps in order. Here we use the same CSV data loaded as `X`. Each step receives the output of the previous step.

### 6.1 Fill and standardize numerical columns

Use the first 900 rows to fit the pipeline. Fill missing age/income values with 0, then standardize each column. Zero filling is used only as a simple example.

In [ ]:
from numcompute.pipeline import Pipeline
from numcompute.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from numcompute.preprocessing import SimpleImputer as Imputer

# numeric_input = X[:, :2].astype(float)
numeric_input = io.load_csv(sample_path, usecols=["age", "income"], dtype=float)
X_train = numeric_input[:900]
X_test = numeric_input[900:]

numeric_pipe = Pipeline([
    ("impute", Imputer(strategy="constant",fill_value=0)),
    ("scale", StandardScaler()),
])

X_train_ready = numeric_pipe.fit_transform(X_train)
print("Training shape:", X_train_ready.shape)
print(X_train_ready[:5])
print("Training column means:", np.round(X_train_ready.mean(axis=0), 6))

### 6.2 Reuse the fitted pipeline

Call `transform` on the remaining 300 rows. This reuses the parameters learned from the first 900 rows and does not fit again.

In [ ]:
X_test_ready = numeric_pipe.transform(X_test)
print("Test shape:", X_test_ready.shape)
print(X_test_ready[:5])

### 6.3 Fill and encode cities

For the categorical column, replace NaN with `Unknown`, then apply one-hot encoding.

In [ ]:
city_pipe = Pipeline([
    ("impute", Imputer(fill_value="Unknown")),
    ("encode", OneHotEncoder()),
])

cities_ready = city_pipe.fit_transform(X[:, 2:3])
print("Encoded city shape:", cities_ready.shape)
print(cities_ready[:5])

## 7. Finite-Difference Gradients and Jacobians

Estimate derivatives by slightly changing each input coordinate. Forward differences compare `f(x+h)` with `f(x)`; central differences compare `f(x+h)` with `f(x-h)`.

Use the first standardized numerical row from the CSV pipeline as the evaluation point. This keeps age and income on comparable scales.

In [ ]:
from numcompute.optim import grad, jacobian

x = X_train_ready[0].copy()
print("Evaluation point:", x)

### 7.1 Gradient of a scalar function

For `f(x) = sum(x**2)`, the analytical gradient is `2*x`. The result has the same shape as the one-dimensional input.

In [ ]:
def f(x):
    return np.sum(x ** 2)

central_gradient = grad(f, x, h=1e-5, method="central")
forward_gradient = grad(f, x, h=1e-5, method="forward")
exact_gradient = 2 * x

print("Central gradient:", central_gradient)
print("Forward gradient:", forward_gradient)
print("Analytical gradient:", exact_gradient)
print("Central result matches:", np.allclose(central_gradient, exact_gradient))

### 7.2 Jacobian of a vector function

For `F(x) = [x[0]**2 + x[1], x[0]*x[1]]`, the analytical Jacobian is `[[2*x[0], 1], [x[1], x[0]]]`. Rows correspond to output components and columns to input coordinates.

In [ ]:
def F(x):
    return np.array([
        x[0] ** 2 + x[1],
        x[0] * x[1],
    ])

central_J = jacobian(F, x, method="central")
forward_J = jacobian(F, x, method="forward")
exact_J = np.array([[2 * x[0], 1], [x[1], x[0]]])

print("Central Jacobian:")
print(central_J)
print("Forward Jacobian:")
print(forward_J)
print("Analytical Jacobian:")
print(exact_J)
print("Shape:", central_J.shape)
print("Central result matches:", np.allclose(central_J, exact_J))

## 8. Benchmark (`benchmarking.py`)

`compare` runs a plain-Python loop version and the NumCompute version of the same computation, checks that they return the same result, and then reports the median time of 7 calls for each. 

The full, reproducible run is `python -m benchmark.run_benchmarks` from the repository root, and its table (copied into the README) is in `benchmark/results.md`.

### 8.1 Loop vs NumCompute

Three cases with very different speed-ups: `mse` on 100,000 random values, `accuracy` on the 3,000 city labels from Section 5, and `rank` on 1,000 of the Section 5 incomes. Lists for the loop side are made before timing, so the conversion is not counted.

In [ ]:
from IPython.display import Markdown, display
from numcompute.benchmarking import compare, environment_info, format_table
from benchmark import loops                       # plain-Python versions

rng = np.random.default_rng(42)
n = 100_000
v_true, v_pred = rng.normal(size=n), rng.normal(size=n)
v_true_list, v_pred_list = v_true.tolist(), v_pred.tolist()
city_true_list, city_pred_list = city_true.tolist(), city_pred.tolist()
income_1k = income[:1_000]                        # Section 5's known incomes
income_1k_list = income_1k.tolist()

cases = [
    (f"metrics.mse n={n:,}",
     lambda: loops.mse_loop(v_true_list, v_pred_list),
     lambda: metrics.mse(v_true, v_pred),
     "Element-wise O(n) in C instead of the interpreter"),
    (f"metrics.accuracy n={city_true.size:,} cities",
     lambda: loops.accuracy_loop(city_true_list, city_pred_list),
     lambda: metrics.accuracy(city_true, city_pred),
     "One comparison per element, done in C"),
    ("rank.rank 1,000 incomes",
     lambda: loops.average_rank_loop(income_1k_list),
     lambda: rank(income_1k),
     "The loop compares every pair, O(n^2); NumPy sorts once"),
]
rows = []
for operation, loop_func, vec_func, notes in cases:
    row = compare(operation, loop_func, vec_func)
    row['notes'] = notes
    rows.append(row)

print(environment_info())
display(Markdown(format_table(rows)))

### 8.2 The result check

A "fast" mse that forgets to divide by the number of samples is rejected before it is timed, so a wrong version can never reach the table.

In [ ]:
try:
    compare("mse, sum only",
            lambda: loops.mse_loop(v_true_list, v_pred_list),
            lambda: np.sum((v_true - v_pred) ** 2))
except ValueError as err:
    print('ValueError:', err)